In [1]:
# ============================================================
# PS S6E5 - exp_20260501_003_lgb_compound_tyrelife_min
# LightGBM + Compound/TyreLife safe interaction features
# ============================================================

In [2]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder

import lightgbm as lgb

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260501_003_lgb_compound_tyrelife_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = BASE / "train.csv"
    TEST_PATH = BASE / "test.csv"
    SUB_PATH = BASE / "sample_submission.csv"

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"

    SEED = 42
    N_SPLITS = 5

    # LightGBM baseline params
    PARAMS = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "learning_rate": 0.03,
        "num_leaves": 64,
        "max_depth": -1,
        "min_child_samples": 40,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "n_estimators": 5000,
        "random_state": SEED,
        "verbosity": -1,
    }


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

print("train shape:", train.shape)
print("test shape :", test.shape)
print("sub shape  :", sub.shape)

print("\ntrain columns:")
print(train.columns.tolist())

print("\ntest columns:")
print(test.columns.tolist())

assert CFG.ID_COL in train.columns
assert CFG.ID_COL in test.columns
assert CFG.TARGET in train.columns
assert CFG.TARGET not in test.columns

feature_cols_raw = [c for c in train.columns if c not in [CFG.ID_COL, CFG.TARGET]]

print("\nraw feature cols:", len(feature_cols_raw))
print(feature_cols_raw)

print("\ntarget distribution:")
print(train[CFG.TARGET].value_counts(dropna=False))
print(train[CFG.TARGET].value_counts(normalize=True, dropna=False))

print("\nCompound values:")
print(train["Compound"].value_counts(dropna=False))
print(test["Compound"].value_counts(dropna=False))

train shape: (439140, 16)
test shape : (188165, 15)
sub shape  : (188165, 2)

train columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']

test columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

raw feature cols: 14
['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

target distribution:
PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64

Compound values:
Compound
MEDIUM          211141
HARD            170518
SOFT             38744
INTERMEDIATE     1

In [5]:
# ============================================================
# Feature Engineering
# ============================================================

def add_compound_tyrelife_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Safe Compound/TyreLife interaction features.

    Important:
    - HARD/MEDIUM/SOFT are dry compounds and have a rough softness order.
    - INTERMEDIATE/WET are wet-weather compounds and should not be forced
      into the same ordinal scale as dry compounds.
    - No Normalized_TyreLife proxy is created.
    - No future stint-ending information is used.
    """
    df = df.copy()

    compound_softness_map = {
        "HARD": 0.0,
        "MEDIUM": 1.0,
        "SOFT": 2.0,
        "INTERMEDIATE": np.nan,
        "WET": np.nan,
    }

    # This is not a physical order. It is only a fallback numerical code.
    compound_full_code_map = {
        "HARD": 0,
        "MEDIUM": 1,
        "SOFT": 2,
        "INTERMEDIATE": 3,
        "WET": 4,
    }

    df["Compound_softness_dry"] = df["Compound"].map(compound_softness_map)
    df["Compound_code_all"] = df["Compound"].map(compound_full_code_map)

    df["Compound_is_dry"] = df["Compound"].isin(["HARD", "MEDIUM", "SOFT"]).astype(int)
    df["Compound_is_wet"] = df["Compound"].isin(["INTERMEDIATE", "WET"]).astype(int)

    df["Compound_is_hard"] = (df["Compound"] == "HARD").astype(int)
    df["Compound_is_medium"] = (df["Compound"] == "MEDIUM").astype(int)
    df["Compound_is_soft"] = (df["Compound"] == "SOFT").astype(int)
    df["Compound_is_intermediate"] = (df["Compound"] == "INTERMEDIATE").astype(int)
    df["Compound_is_wet_tyre"] = (df["Compound"] == "WET").astype(int)

    # Dry compound softness interactions
    # INTERMEDIATE/WET become NaN here by design.
    for col in ["TyreLife", "Cumulative_Degradation", "RaceProgress"]:
        df[f"dry_softness_x_{col}"] = df["Compound_softness_dry"] * df[col]

    # Dry/wet regime interactions
    for col in ["TyreLife", "Cumulative_Degradation", "RaceProgress"]:
        df[f"is_dry_x_{col}"] = df["Compound_is_dry"] * df[col]
        df[f"is_wet_x_{col}"] = df["Compound_is_wet"] * df[col]

    # Compound-specific interactions
    compound_flags = [
        "Compound_is_hard",
        "Compound_is_medium",
        "Compound_is_soft",
        "Compound_is_intermediate",
        "Compound_is_wet_tyre",
    ]

    for flag_col in compound_flags:
        df[f"{flag_col}_x_TyreLife"] = df[flag_col] * df["TyreLife"]
        df[f"{flag_col}_x_Cumulative_Degradation"] = df[flag_col] * df["Cumulative_Degradation"]
        df[f"{flag_col}_x_RaceProgress"] = df[flag_col] * df["RaceProgress"]

    # Stint interactions
    df["Stint_x_TyreLife"] = df["Stint"] * df["TyreLife"]
    df["Stint_x_Cumulative_Degradation"] = df["Stint"] * df["Cumulative_Degradation"]
    df["Stint_x_RaceProgress"] = df["Stint"] * df["RaceProgress"]
    df["Stint_x_dry_softness"] = df["Stint"] * df["Compound_softness_dry"]
    df["Stint_x_is_wet"] = df["Stint"] * df["Compound_is_wet"]

    return df


train_fe = add_compound_tyrelife_features(train)
test_fe = add_compound_tyrelife_features(test)

feature_cols = [c for c in train_fe.columns if c not in [CFG.ID_COL, CFG.TARGET]]

added_features = [c for c in feature_cols if c not in feature_cols_raw]

print("\nadded features:", len(added_features))
for c in added_features:
    print(c)

print("\ntotal feature cols:", len(feature_cols))


added features: 38
Compound_softness_dry
Compound_code_all
Compound_is_dry
Compound_is_wet
Compound_is_hard
Compound_is_medium
Compound_is_soft
Compound_is_intermediate
Compound_is_wet_tyre
dry_softness_x_TyreLife
dry_softness_x_Cumulative_Degradation
dry_softness_x_RaceProgress
is_dry_x_TyreLife
is_wet_x_TyreLife
is_dry_x_Cumulative_Degradation
is_wet_x_Cumulative_Degradation
is_dry_x_RaceProgress
is_wet_x_RaceProgress
Compound_is_hard_x_TyreLife
Compound_is_hard_x_Cumulative_Degradation
Compound_is_hard_x_RaceProgress
Compound_is_medium_x_TyreLife
Compound_is_medium_x_Cumulative_Degradation
Compound_is_medium_x_RaceProgress
Compound_is_soft_x_TyreLife
Compound_is_soft_x_Cumulative_Degradation
Compound_is_soft_x_RaceProgress
Compound_is_intermediate_x_TyreLife
Compound_is_intermediate_x_Cumulative_Degradation
Compound_is_intermediate_x_RaceProgress
Compound_is_wet_tyre_x_TyreLife
Compound_is_wet_tyre_x_Cumulative_Degradation
Compound_is_wet_tyre_x_RaceProgress
Stint_x_TyreLife
Stint_

In [6]:
# ============================================================
# Column types
# ============================================================

# Keep baseline categorical policy close to 001/002 logs:
# Year/LapNumber/PitStop/Position/Stint are treated as categorical.
cat_cols = [
    "Driver",
    "Compound",
    "Race",
    "Year",
    "LapNumber",
    "PitStop",
    "Position",
    "Stint",
]

cat_cols = [c for c in cat_cols if c in feature_cols]
num_cols = [c for c in feature_cols if c not in cat_cols]

print("\ncat_cols:", len(cat_cols), cat_cols)
print("num_cols:", len(num_cols), num_cols)


cat_cols: 8 ['Driver', 'Compound', 'Race', 'Year', 'LapNumber', 'PitStop', 'Position', 'Stint']
num_cols: 44 ['TyreLife', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'Compound_softness_dry', 'Compound_code_all', 'Compound_is_dry', 'Compound_is_wet', 'Compound_is_hard', 'Compound_is_medium', 'Compound_is_soft', 'Compound_is_intermediate', 'Compound_is_wet_tyre', 'dry_softness_x_TyreLife', 'dry_softness_x_Cumulative_Degradation', 'dry_softness_x_RaceProgress', 'is_dry_x_TyreLife', 'is_wet_x_TyreLife', 'is_dry_x_Cumulative_Degradation', 'is_wet_x_Cumulative_Degradation', 'is_dry_x_RaceProgress', 'is_wet_x_RaceProgress', 'Compound_is_hard_x_TyreLife', 'Compound_is_hard_x_Cumulative_Degradation', 'Compound_is_hard_x_RaceProgress', 'Compound_is_medium_x_TyreLife', 'Compound_is_medium_x_Cumulative_Degradation', 'Compound_is_medium_x_RaceProgress', 'Compound_is_soft_x_TyreLife', 'Compound_is_soft_x_Cumulative_Degradation', 'Compound_is_soft_x_R

In [7]:
# ============================================================
# Prepare X/y
# ============================================================

X = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()

# S6E5 target is numeric 0.0/1.0, so do not string-map it.
assert train_fe[CFG.TARGET].notna().all()
assert set(train_fe[CFG.TARGET].dropna().unique()).issubset({0, 1, 0.0, 1.0})
y = train_fe[CFG.TARGET].astype(int).values

# Ordinal encode categorical columns for LightGBM
X_enc = X.copy()
X_test_enc = X_test.copy()

if cat_cols:
    enc = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=-2,
    )
    X_enc[cat_cols] = enc.fit_transform(
        X_enc[cat_cols].astype("string").fillna("__MISSING__")
    )
    X_test_enc[cat_cols] = enc.transform(
        X_test_enc[cat_cols].astype("string").fillna("__MISSING__")
    )

print("\nX_enc shape:", X_enc.shape)
print("X_test_enc shape:", X_test_enc.shape)

print("\nAny inf in numeric?")
num_values = X_enc[num_cols].replace([np.inf, -np.inf], np.nan)
print("train numeric missing count:", num_values.isna().sum().sum())


X_enc shape: (439140, 52)
X_test_enc shape: (188165, 52)

Any inf in numeric?
train numeric missing count: 93685


In [8]:
# ============================================================
# CV Training
# ============================================================

oof = np.zeros(len(train_fe), dtype=float)
pred = np.zeros(len(test_fe), dtype=float)

fold_scores = []
fold_best_iterations = []
feature_importance_list = []

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_enc, y), 1):
    print("\n" + "=" * 80)
    print(f"Fold {fold}")
    print("=" * 80)

    X_tr = X_enc.iloc[tr_idx]
    X_va = X_enc.iloc[va_idx]
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    model = lgb.LGBMClassifier(**CFG.PARAMS)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(200),
        ],
        categorical_feature=cat_cols if cat_cols else "auto",
    )

    va_pred = model.predict_proba(X_va)[:, 1]
    te_pred = model.predict_proba(X_test_enc)[:, 1]

    oof[va_idx] = va_pred
    pred += te_pred / CFG.N_SPLITS

    auc = roc_auc_score(y_va, va_pred)
    fold_scores.append(float(auc))
    fold_best_iterations.append(int(model.best_iteration_))

    print(f"fold {fold} AUC: {auc:.8f}")
    print(f"fold {fold} best_iteration: {model.best_iteration_}")

    fi = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_,
        "fold": fold,
    })
    feature_importance_list.append(fi)


cv_auc = roc_auc_score(y, oof)

print("\n" + "=" * 80)
print("CV RESULT")
print("=" * 80)
print("CV AUC:", cv_auc)
print("fold_scores:", fold_scores)
print("mean:", np.mean(fold_scores))
print("std :", np.std(fold_scores))
print("fold_best_iterations:", fold_best_iterations)


Fold 1
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.940229
[400]	valid_0's auc: 0.941802
[600]	valid_0's auc: 0.942105
[800]	valid_0's auc: 0.942017
Early stopping, best iteration is:
[724]	valid_0's auc: 0.942241
fold 1 AUC: 0.94224141
fold 1 best_iteration: 724

Fold 2
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.938579
[400]	valid_0's auc: 0.940054
[600]	valid_0's auc: 0.940181
[800]	valid_0's auc: 0.94022
Early stopping, best iteration is:
[715]	valid_0's auc: 0.940403
fold 2 AUC: 0.94040282
fold 2 best_iteration: 715

Fold 3
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.939821
[400]	valid_0's auc: 0.941312
[600]	valid_0's auc: 0.941464
[800]	valid_0's auc: 0.941499
Early stopping, best iteration is:
[750]	valid_0's auc: 0.941624
fold 3 AUC: 0.94162367
fold 3 best_iteration: 750

Fold 4
Training until validation scores don't improve for 200 rounds
[200]	valid_0's 

In [9]:
# ============================================================
# Save artifacts
# ============================================================

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred)

sub_out = sub.copy()
pred_col = [c for c in sub_out.columns if c != CFG.ID_COL][0]
sub_out[pred_col] = pred
sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub_out.to_csv(sub_path, index=False)

feature_df = pd.DataFrame({
    "feature": feature_cols,
    "is_added": [c in added_features for c in feature_cols],
    "is_categorical": [c in cat_cols for c in feature_cols],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

fi_all = pd.concat(feature_importance_list, axis=0, ignore_index=True)
fi_summary = (
    fi_all
    .groupby("feature", as_index=False)["importance"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

fi_all.to_csv(CFG.OUTDIR / "feature_importance_by_fold.csv", index=False)
fi_summary.to_csv(CFG.OUTDIR / "feature_importance.csv", index=False)

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "data": {
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "target_distribution": {
            str(k): int(v)
            for k, v in train[CFG.TARGET].value_counts(dropna=False).to_dict().items()
        },
    },
    "features": {
        "raw_feature_count": len(feature_cols_raw),
        "total_feature_count": len(feature_cols),
        "added_feature_count": len(added_features),
        "categorical_features": cat_cols,
        "numerical_features": num_cols,
        "added_features": added_features,
        "notes": [
            "Compound remains categorical.",
            "Dry compound ordinal is used only for HARD/MEDIUM/SOFT.",
            "INTERMEDIATE/WET are treated as wet-weather compounds.",
            "No Normalized_TyreLife proxy is used.",
            "No future-looking stint normalization is used.",
        ],
    },
    "model": {
        "family": "LightGBM",
        "params": CFG.PARAMS,
        "cv": {
            "scheme": "StratifiedKFold",
            "n_splits": CFG.N_SPLITS,
            "shuffle": True,
            "random_state": CFG.SEED,
        },
    },
    "result": {
        "cv_auc": float(cv_auc),
        "fold_scores": fold_scores,
        "fold_best_iterations": fold_best_iterations,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
    },
    "artifacts": {
        "oof": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "submission": str(sub_path),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "feature_importance": str(CFG.OUTDIR / "feature_importance.csv"),
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved artifacts to:", CFG.OUTDIR)
print("Submission:", sub_path)

display(fi_summary.head(40))
display(sub_out.head())


Saved artifacts to: /kaggle/working/exp_20260501_003_lgb_compound_tyrelife_min
Submission: /kaggle/working/exp_20260501_003_lgb_compound_tyrelife_min/submission_exp_20260501_003_lgb_compound_tyrelife_min.csv


,index,feature,mean,std
26,26,Driver,17320.0,599.807886
27,27,LapNumber,8053.6,418.838036
33,33,Race,5352.4,303.668899
31,31,Position,3110.6,213.217729
29,29,LapTime_Delta,1105.4,51.988460
38,38,Stint_x_TyreLife,1015.2,18.376616
41,41,TyreLife,848.2,18.335757
37,37,Stint_x_RaceProgress,731.4,40.172130
28,28,LapTime (s),720.4,40.857068
32,32,Position_Change,656.4,36.294628


,id,PitNextLap
0,439140,0.004132
1,439141,0.003915
2,439142,0.008310
3,439143,0.180279
4,439144,0.748822
